# Reproduce the qvunex numbers on a free T4

This notebook repeats two measurements from the [qvunex README](https://github.com/qaisermehdi3-coder/qvunex#if-you-run-your-own-model-server-vllm), live, on whatever Colab gives you:

1. **Scale-to-zero:** how much of one cold wake's billed time is the boot, and what that does to the cost per request.
2. **Busy vs idle:** per-request speed against whole-server speed, 8 requests one at a time against 8 at once.

It uses `qvunex_single.py` (one file, standard library only) and vLLM's own `/metrics` counters. Nothing leaves the machine.

**Runtime:** this notebook asks for a T4 GPU. If Colab gives you a CPU runtime, go to Runtime → Change runtime type → T4 GPU **before** running anything (changing it later wipes the install).

Run the three code cells in order. Your numbers will differ from ours: different host, different moment. That spread is part of the point.

In [ ]:
# 1. Install (about 5 minutes). Red "dependency conflict" lines about other packages are normal.
!pip install -q vllm==0.27.1 openai && pip uninstall -y -q torchaudio
!wget -q -O qvunex_single.py https://raw.githubusercontent.com/qaisermehdi3-coder/qvunex/main/qvunex_single.py
!python3 qvunex_single.py --selftest | grep -E "passed|FAIL"

In [ ]:
# 2. One cold wake, priced like a cloud that bills from creation (about 3 minutes)
import subprocess, time, urllib.request, os
os.environ["QVUNEX_PATH"] = "/content/wake.jsonl"
M = "Qwen/Qwen2.5-0.5B-Instruct"
RATE = 1.72   # dollars per hour for the server; change to your own
t0 = time.time()
srv = subprocess.Popen(f"vllm serve {M} --dtype half --port 8000 > server.log 2>&1", shell=True)
up = False
for i in range(72):
    time.sleep(5)
    if srv.poll() is not None:
        print("SERVER CRASHED"); print(open("server.log").read()[-1500:]); break
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models"); up = True; break
    except Exception: pass
if up:
    print("server answered after about", round(time.time() - t0), "s")
    subprocess.run("curl -s localhost:8000/metrics > before.txt", shell=True)
    import qvunex_single as q
    from openai import OpenAI
    c = q.wrap(OpenAI(base_url="http://localhost:8000/v1", api_key="x"))
    for i in range(8):
        c.chat.completions.create(model=M, max_tokens=64, temperature=0,
            messages=[{"role": "user", "content": f"Write a short story about a lamp, number {i}."}])
    subprocess.run("curl -s localhost:8000/metrics > after.txt", shell=True)
    billed = time.time() - t0
    q.reconcile("before.txt", "after.txt", path="/content/wake.jsonl", gpu_rate=RATE, window=billed)

In [ ]:
# 3. Busy vs idle: 8 requests one at a time, then 8 at once, three rounds (about 1 minute)
import random, io
from concurrent.futures import ThreadPoolExecutor
words = "lamp river stone cloud market engine paper window garden signal".split()
def call(i):
    text = f"id {random.random()} " + " ".join(random.choice(words) for _ in range(400))
    c.chat.completions.create(model=M, max_tokens=64, temperature=0,
        messages=[{"role": "user", "content": "Summarise: " + text}])
fmt = lambda v: "unknown" if v is None else f"{v:,.1f}"
def window(name, workers):
    q.configure(path=f"/content/{name}.jsonl")
    subprocess.run(f"curl -s localhost:8000/metrics > {name}_b.txt", shell=True); t = time.time()
    with ThreadPoolExecutor(workers) as ex: list(ex.map(call, range(8)))
    w = time.time() - t
    subprocess.run(f"curl -s localhost:8000/metrics > {name}_a.txt", shell=True)
    r = q.reconcile(f"{name}_b.txt", f"{name}_a.txt", path=f"/content/{name}.jsonl", window=w, out=io.StringIO())
    s = r["speed"]
    print(f"{name}: {workers} at once, meter vs server gaps {r['gaps']}, per-request decode {fmt(s['decode'])} tok/s, whole server {fmt(s['whole_output'])} tok/s")
for k in range(3):
    window(f"one{k}", 1); window(f"eight{k}", 8)